In [ ]:
import pathlib
import re
import textwrap

import pandas as pd
from IPython.display import Image, clear_output, display

cwd = pathlib.Path.cwd()
repo_root = next(p for p in [cwd, *cwd.parents] if (p / 'docs' / 'eval_protocol.md').exists())

judgments_path = repo_root / 'docs' / 'relevance_judgments.csv'
protocol_path = repo_root / 'docs' / 'eval_protocol.md'
image_dir = repo_root / 'data' / 'raw' / 'train'

In [ ]:
text = protocol_path.read_text()
criteria = dict(re.findall(r'^\*\*([ABCD]\d)\. `[^`]+`\*\*\n- \*\*Relevant:\*\* (.*?)(?=\n- \*\*|\n\n)', text, flags=re.M | re.S))
criteria = {query_id: ' '.join(criterion.split()) for query_id, criterion in criteria.items()}

assert len(criteria) == 20, f"expected 20 criteria, got {len(criteria)}"

In [ ]:
judgments_df = pd.read_csv(judgments_path, dtype={'score': 'Int64', 'reason': 'string'})
todo = judgments_df[judgments_df['score'].isna()].sort_values(['query_id', 'judge_order']).index

for row_i in todo:
    row = judgments_df.loc[row_i]
    n_in_query = (judgments_df['query_id'] == row['query_id']).sum()
    
    clear_output(wait=True)
    print(f"{judgments_df['score'].notna().sum()} / {len(judgments_df)} judged")
    print(f"\n{row['query_id']}. '{row['query']}', image {row['judge_order']} of {n_in_query}")
    print(textwrap.fill(f"Relevant: {criteria[row['query_id']]}", 100))
    display(Image(filename=str(image_dir / row['file_name']), width=500))
    
    score = input('score 0 / 1 / 2, or q to stop: ').strip()
    while score not in {'0', '1', '2', 'q'}:
        score = input('score must be 0, 1 or 2, or q to stop: ').strip()
    
    if score == 'q':
        break
    
    judgments_df.loc[row_i, 'score'] = int(score)
    judgments_df.loc[row_i, 'reason'] = input('reason: ').strip()
    
    # save after every judgment so a crash or a break loses nothing
    judgments_df.to_csv(judgments_path, index=False)

print(f"{judgments_df['score'].notna().sum()} / {len(judgments_df)} judged")